# Fine-tuning MobileNetV3-Large para Classificação de Folhas de Soja

**Hiperparâmetros ótimos** (Bayesian Search):
- Dropout FC1: 0.4730 | Dropout FC2: 0.3255
- FC1: 512 | FC2: 512
- Optimizer: Adam (lr=0.00483)
- Batch Size: 128

In [ ]:
import os, torch, numpy as np, torch.nn as nn, torch.optim as optim
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping, ModelCheckpoint
from ignite.metrics import Accuracy, Loss
from dotenv import load_dotenv
load_dotenv()

DATASET_PATH = os.getenv("DATASET_PATH", "/caminho/para/DADOS-DIVIDIDOS")
PRETRAINED_WEIGHTS = os.getenv("MOBILENETV3_PRETRAINED", "/caminho/para/models/mobilenet_v3_large-model-84.pth")
RESULTS_DIR = os.getenv("RESULTS_DIR", "./results/mobilenetv3")
CHECKPOINT_DIR = os.getenv("CHECKPOINT_DIR", "./checkpoints/mobilenetv3")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 128; NUM_EPOCHS = 100
LEARNING_RATE = 0.004832285174962019
DROPOUT1 = 0.4730438988122177
DROPOUT2 = 0.3254939388545582
FC1_NEURONS = 512; FC2_NEURONS = 512
NUM_CLASSES = 2; FEATURE_EXTRACT = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Data Augmentation e Carregamento

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224), transforms.CenterCrop(224),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224), transforms.CenterCrop(224),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATASET_PATH, x), data_transforms[x]) for x in ['train', 'val', 'test']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x=='train'), num_workers=4) for x in ['train', 'val', 'test']}
for s in ['train','val','test']: print(f"{s}: {len(image_datasets[s])} imagens")

## 2. Transfer Learning MobileNetV3

In [ ]:
def set_parameter_requires_grad(model, fe):
    if fe:
        for p in model.parameters(): p.requires_grad = False

model = models.mobilenet_v3_large(pretrained=False)
set_parameter_requires_grad(model, FEATURE_EXTRACT)

state_dict = torch.load(PRETRAINED_WEIGHTS, map_location=device)
del state_dict['classifier.3.weight']
del state_dict['classifier.3.bias']
model.load_state_dict(state_dict, strict=False)

num_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT1),
    nn.Linear(num_features, FC1_NEURONS), nn.ReLU(),
    nn.Dropout(p=DROPOUT2),
    nn.Linear(FC1_NEURONS, FC2_NEURONS), nn.ReLU(),
    nn.Linear(FC2_NEURONS, NUM_CLASSES),
    nn.Softmax(dim=1),
)
model = model.to(device)
print(f"Classificador:\n{model.classifier}")

## 3-4. Treinamento com Ignite

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

def train_step(engine, batch):
    model.train(); x, y = batch[0].to(device), batch[1].to(device)
    optimizer.zero_grad(); out = model(x); loss = criterion(out, y)
    loss.backward(); optimizer.step(); return loss.item(), out, y

def eval_step(engine, batch):
    model.eval()
    with torch.no_grad(): x, y = batch[0].to(device), batch[1].to(device); return model(x), y

trainer = Engine(train_step); evaluator = Engine(eval_step)
Accuracy().attach(evaluator, 'accuracy'); Loss(criterion).attach(evaluator, 'loss')
history = {'val_loss': [], 'val_accuracy': []}

def score_fn(e): return e.state.metrics['accuracy']
evaluator.add_event_handler(Events.COMPLETED, EarlyStopping(patience=10, score_function=score_fn, trainer=trainer))
evaluator.add_event_handler(Events.COMPLETED, ModelCheckpoint(CHECKPOINT_DIR, 'best', n_saved=1, score_function=score_fn, score_name='accuracy', require_empty=False), {'model': model})

@trainer.on(Events.EPOCH_COMPLETED)
def log(engine):
    evaluator.run(dataloaders['val']); m = evaluator.state.metrics
    history['val_loss'].append(m['loss']); history['val_accuracy'].append(m['accuracy'])
    scheduler.step(m['loss']); print(f"Época {engine.state.epoch}/{NUM_EPOCHS} — Val Acc: {m['accuracy']:.4f}")

## 5. Treinamento

In [ ]:
trainer.run(dataloaders['train'], max_epochs=NUM_EPOCHS)
print("Finalizado!")

## 6. Avaliação e Curvas

In [ ]:
best_path = os.path.join(CHECKPOINT_DIR, os.listdir(CHECKPOINT_DIR)[-1])
model.load_state_dict(torch.load(best_path, map_location=device))
evaluator.run(dataloaders['test'])
t = evaluator.state.metrics
print(f"TESTE — Accuracy: {t['accuracy']:.4f} | Loss: {t['loss']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['val_loss'], color='#e74c3c'); axes[0].set_title('Loss'); axes[0].grid(True, alpha=0.3)
axes[1].plot(history['val_accuracy'], color='#2ecc71'); axes[1].set_title('Accuracy'); axes[1].grid(True, alpha=0.3)
plt.suptitle('MobileNetV3 — Curvas de Aprendizagem', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'mobilenetv3_learning_curves.png'), dpi=150); plt.show()